# 热镀锌卷序优化示例

这个示例面向钢铁企业的冷轧、热轧和热镀锌计划人员，说明如何把连续热镀锌线（CGL/HDG）的合同卷排序问题表达成一个清晰、可解释、可求解的优化模型。

这里关注的不是生产系统集成，而是一个更直接的问题：求解器是否能覆盖热镀锌排程的业务约束和现场痛点，以及 OptAgent 如何让这类模型更容易声明、调试和解释。

## 热镀锌业务痛点

热镀锌线不是把合同卷随便排成一列。它是一条连续、高节奏、强工艺耦合的产线，排程质量会直接影响稳定生产、表面质量、换辊节奏和交付效率。

典型痛点包括：

- **锌层 campaign 难稳定**：GI、GA 或不同锌层重量频繁切换，会带来锌锅、气刀、镀层控制和质量判定压力。
- **退火温度跳变影响炉区稳定**：不同钢种、不同退火制度穿插时，炉温设定和带钢实际温度更难平稳过渡。
- **宽度和厚度跳跃影响板形与穿带风险**：宽度大幅上跳、厚薄规格频繁交替，会增加辊系调整、板形控制和跑偏风险。
- **外板质量窗口需要保护**：汽车外板或高表面要求材料通常希望成组生产，避免被普通材料或不稳定过渡打散。
- **薄规格和后处理切换需要连续性**：薄规格材料、钝化、涂油等后处理路线频繁切换，会增加操作负担和质量波动。
- **人工调整难以全局平衡**：计划员可以凭经验修局部顺序，但很难同时权衡锌层、宽度、厚度、温度、外板和后处理等多维目标。

因此，一个有价值的排程模型不只是给出“可行顺序”，还应能解释为什么这个顺序更适合生产。

## 求解器能做到什么

对热镀锌卷序问题，求解器的核心价值是把人工难以穷举的排列空间交给算法搜索。即使只有几十个合同卷，可能顺序数量也会快速膨胀，靠人工逐个比较不现实。

一个合适的求解器通常可以做到：

- 把合同卷顺序建成一个排列变量，而不是手工枚举所有可能顺序；
- 根据相邻卷的宽度、厚度、退火温度、锌层和工艺路线计算过渡成本；
- 从人工草案或 APS 初始顺序出发，搜索更低成本的序列；
- 在多条规则互相冲突时做权衡，例如为了锌层连续性接受少量宽度跳跃；
- 输出总成本和分项规则成本，帮助计划员判断优化结果是否符合现场经验。

换句话说，求解器可以承担“在巨大排列空间中找更好方案”的工作，让计划员把精力放在规则是否合理、结果是否可执行上。

## OptAgent 能做得更好的地方

热镀锌排程的难点不只在求解，还在建模表达。很多现场规则并不天然是一个简单线性公式，例如外板块保护、换辊风险、薄规格连续性、后处理切换、退火温度过渡等，往往需要和业务人员反复校准。

OptAgent 在这个示例中强调三点：

- **建模声明更清晰**：用 `sequence_var` 直接声明合同卷排列，用 `external_call` 直接接入 Python 业务评分函数，不需要先把所有规则硬翻译成复杂数学式。
- **规则解释更自然**：宽度平滑、厚度平滑、退火温度、锌层切换、外板打断、后处理切换等成本可以保留为独立命名项，便于和计划、工艺、质量人员沟通。
- **从业务草案开始优化**：模型可以从人工或 APS 给出的 incumbent sequence 出发，做局部修补和改进，而不是给现场一个难以理解的全新黑箱顺序。
- **适合逐步工程化**：示例用简洁的黑箱评分展示能力，后续可以逐步加入更正式的数据接口、参数标定、结果审计和生产输出表。

这使得 OptAgent 更适合做热镀锌这类“规则多、解释要求高、现场需要逐步接受”的排程优化。

## 示例如何阅读

推荐入口：

```text
src/hot_dip_galvanizing_model.ipynb
```

这个 notebook 使用内联的订单表、材料表、机组表和初始计划表，不依赖 SQLite，并用 `pandas.DataFrame` 在 Jupyter 中逐表展示。它会展示：

1. 如何用订单表、材料表、机组表和初始计划表描述热镀锌业务输入；
2. 如何把多张业务表 join 成模型需要的待排合同卷；
3. 如何把锌层、宽度、厚度、退火温度、外板、薄规格、后处理和换辊风险写成可解释的规则成本；
4. 如何用 `sequence_var` 声明卷序变量；
5. 如何用 `external_call` 接入业务评分函数；
6. 如何运行启发式搜索并查看优化前后的规则成本变化。
